# Notebook pour une baseline "plafond" (QA extractif): deepset/roberta-base-squad2

## 1) Imports

In [1]:
import os
import json
import ast
import collections
import re
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    set_seed,
)
import torch


/home/rayane/Desktop/Paris Cité/M1/PPD/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Paramètres (modèle, chemins, tailles)

Cette cellule définit les paramètres utilisés dans tout le notebook.

- **Entrées :** aucune.
- **Ce que fait la cellule :**
  - choisit le modèle de base (`deepset/roberta-base-squad2`),
  - donne les chemins des fichiers `train.csv` et `validation.csv`,
  - fixe les tailles importantes :
    - `MAX_SEQ_LENGTH` : longueur max (question + contexte),
    - `MAX_QUERY_LENGTH` : longueur max de la question,
    - `DOC_STRIDE` : chevauchement quand on coupe les contextes longs,
    - `N_BEST_SIZE` / `MAX_ANSWER_LENGTH` : paramètres pour reconstruire la réponse.
- **Sorties :** aucune (ce sont des constantes globales).

In [2]:
MODEL_NAME = "deepset/roberta-base-squad2"

DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"


MAX_SEQ_LENGTH = 512
DOC_STRIDE = 128
MAX_QUERY_LENGTH = 128
N_BEST_SIZE = 20
MAX_ANSWER_LENGTH = 30

SEED = 42

## 3) Fonctions de nettoyage du dataset

### `parse_answers_field(x)`
- **Entrée :** `x` = la valeur de la colonne `answers` (une chaîne de texte).
- **Ce que fait la fonction :**
  - nettoie le texte `answers` (enlevant la forme `array([...], dtype=...)`),
  - transforme la chaîne en dictionnaire Python (`{"text": [...], "answer_start": [...]}`),
  - convertit le tout en listes Python propres.
- **Sortie :** un dictionnaire au format SQuAD : `{"text": [...], "answer_start": [...]}`.

### `normalize_dataset(example)`
- **Entrée :** `example` = une ligne du dataset (dict).
- **Ce que fait la fonction :**
  - applique `parse_answers_field` sur la colonne `answers`,
  - convertit `id` en string (important pour le calcul des métriques).
- **Sortie :** la ligne `example` corrigée.

### `_truncate_questions(questions, tokenizer)`
- **Entrées :**
  - `questions` = liste de questions,
  - `tokenizer` = tokenizer du modèle.
- **Ce que fait la fonction :**
  - coupe les questions trop longues (max `MAX_QUERY_LENGTH` tokens),
  - reconvertit en texte.
- **Sortie :** une liste de questions raccourcies.

In [3]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example

def _truncate_questions(questions, tokenizer):
    q_tok = tokenizer(
        questions,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_QUERY_LENGTH,
    )
    return tokenizer.batch_decode(
        q_tok["input_ids"],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

## 4) Préparation des données pour l'entraînement (train)

### `prepare_train_features(examples, tokenizer)`
- **Entrées :**
  - `examples` = un batch d’exemples (questions, contextes, answers),
  - `tokenizer` = tokenizer du modèle.
- **Ce que fait la fonction :**
  1) coupe les questions trop longues (avec `_truncate_questions`),
  2) tokenise (question, contexte) en gérant les contextes longs :
     - si le contexte dépasse `MAX_SEQ_LENGTH`, il est coupé en plusieurs morceaux (fenêtres),
  3) transforme la réponse gold (position caractères dans le texte) en **positions de tokens** :
     - `start_positions` et `end_positions`,
  4) si la réponse n’est pas dans une fenêtre, met `(0,0)` (token CLS).
- **Sortie :** un dictionnaire tokenisé contenant :
  - `input_ids`, `attention_mask`, ...
  - `start_positions`, `end_positions` (labels pour apprendre).

In [4]:
def prepare_train_features(examples, tokenizer):
    questions = _truncate_questions(examples["question"], tokenizer)

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers = examples["answers"][sample_idx]

       
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answers["answer_start"][0]
        answer_text = answers["text"][0]
        end_char = start_char + len(answer_text)

        sequence_ids = tokenized.sequence_ids(i)

        
        context_start = 0
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1

        
        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(0)
            end_positions.append(0)
            continue

        
        token_start = context_start
        while token_start <= context_end and offsets[token_start][0] <= start_char:
            token_start += 1
        start_positions.append(token_start - 1)

        
        token_end = context_end
        while token_end >= context_start and offsets[token_end][1] >= end_char:
            token_end -= 1
        end_positions.append(token_end + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

## 5) Préparation des données pour la validation (val)

### `prepare_validation_features(examples, tokenizer)`
- **Entrées :**
  - `examples` = un batch d’exemples validation,
  - `tokenizer` = tokenizer du modèle.
- **Ce que fait la fonction :**
  1) coupe les questions trop longues,
  2) tokenise (question, contexte) avec découpage en fenêtres si besoin,
  3) ajoute `example_id` pour relier chaque fenêtre à l’exemple original,
  4) garde `offset_mapping` uniquement pour les tokens du contexte (le reste = `None`).
- **Sortie :** des features de validation contenant :
  - `example_id`,
  - `offset_mapping`,
  - les champs tokenisés (`input_ids`, `attention_mask`, ...).

In [5]:
def prepare_validation_features(examples, tokenizer):
    questions = _truncate_questions(examples["question"], tokenizer)

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")

   
    tokenized["example_id"] = []


    new_offset_mapping = []
    for i, offsets in enumerate(tokenized["offset_mapping"]):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(str(examples["id"][sample_idx]))

        sequence_ids = tokenized.sequence_ids(i)
        cleaned_offsets = []
        for k, off in enumerate(offsets):
            if sequence_ids[k] == 1:
                cleaned_offsets.append(off)
            else:
                cleaned_offsets.append(None)
        new_offset_mapping.append(cleaned_offsets)

    tokenized["offset_mapping"] = new_offset_mapping
    return tokenized


## 6) Reconstruction des réponses texte (post-processing)

### `postprocess_qa_predictions(examples, features, raw_predictions, tokenizer, ...)`
- **Entrées :**
  - `examples` = dataset validation original (avec `id`, `context`),
  - `features` = fenêtres tokenisées (avec `example_id` + `offset_mapping`),
  - `raw_predictions` = logits du modèle : `(start_logits, end_logits)`,
  - `tokenizer` = tokenizer (pas indispensable ici, mais gardé pour cohérence).
- **Ce que fait la fonction :**
  1) regroupe toutes les fenêtres qui appartiennent au même exemple (`example_id`),
  2) dans chaque fenêtre, cherche les meilleurs débuts et fins de réponse (top `N_BEST_SIZE`),
  3) teste plusieurs spans possibles, garde ceux qui sont valides,
  4) choisit le span avec le meilleur score,
  5) extrait le texte final depuis le `context` original.
- **Sortie :** un dictionnaire :
  - `{"id_exemple": "texte_réponse", ...}`

In [6]:
def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    tokenizer,
    n_best_size = N_BEST_SIZE,
    max_answer_length= MAX_ANSWER_LENGTH,
):

    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {str(k): i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, f in enumerate(features):
        features_per_example[f["example_id"]].append(i)

    predictions = {}

    for example_id, feature_indices in features_per_example.items():
        example_index = example_id_to_index[example_id]
        context = examples["context"][example_index]

        valid_answers = []
        for fi in feature_indices:
            start_logits = all_start_logits[fi]
            end_logits = all_end_logits[fi]
            offsets = features[fi]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offsets) or end_index >= len(offsets):
                        continue
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if end_index < start_index:
                        continue
                    length = end_index - start_index + 1
                    if length > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]
                    score = float(start_logits[start_index] + end_logits[end_index])
                    text = context[start_char:end_char]
                    valid_answers.append({"score": score, "text": text})

        if len(valid_answers) == 0:
            predictions[example_id] = ""
        else:
            best = max(valid_answers, key=lambda x: x["score"])
            predictions[example_id] = best["text"]

    return predictions

## 7) Chargement des données et du modèle

Cette cellule charge les CSV, nettoie le dataset, et charge le modèle.

- **Entrées :**
  - `data/train.csv`
  - `data/validation.csv`
- **Ce que fait la cellule :**
  1) fixe la graine (`set_seed`) pour reproductibilité,
  2) charge les CSV avec `load_dataset`,
  3) applique `normalize_dataset` à toutes les lignes (answers + id),
  4) charge le tokenizer et le modèle QA extractif (`deepset/roberta-base-squad2`).
- **Sorties :**
  - `ds` : dataset HuggingFace (train + validation),
  - `tokenizer` : tokenizer RoBERTa,
  - `model` : modèle QA prêt à fine-tuner.

In [7]:
set_seed(SEED)
ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})

# Normalize answers field
ds = ds.map(normalize_dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)


## 8) Tokenisation et création des features (train / validation)

Cette cellule transforme les données brutes en “features” (données tokenisées) utilisables par le modèle.

### Pourquoi on crée 3 jeux de features ?
On a besoin de **deux types de validation** :
- une validation pour calculer la **loss** (donc il faut les labels `start_positions` / `end_positions`),
- une validation pour calculer **EM/F1** (donc il faut `example_id` et `offset_mapping` pour reconstruire la réponse texte).

### Ce que fait la cellule

1) **`train_features`**
- **Entrée :** `ds["train"]`
- **Fonction appelée :** `prepare_train_features`
- **But :** créer des entrées tokenisées + les labels (`start_positions`, `end_positions`) pour entraîner le modèle.
- **Sortie :** `train_features`

2) **`val_features_for_loss`**
- **Entrée :** `ds["validation"]`
- **Fonction appelée :** `prepare_train_features` (oui, la même que pour train)
- **But :** permettre au Trainer de calculer `eval_loss` à chaque epoch (car il faut des labels).
- **Sortie :** `val_features_for_loss`

3) **`val_features_for_pred`**
- **Entrée :** `ds["validation"]`
- **Fonction appelée :** `prepare_validation_features`
- **But :** garder `example_id` + `offset_mapping` pour reconstruire les réponses en texte, puis calculer **Exact Match** et **F1**.
- **Sortie :** `val_features_for_pred`

In [8]:
# Tokenize
train_features = ds["train"].map(
    lambda x: prepare_train_features(x, tokenizer),
    batched=True,
    remove_columns=ds["train"].column_names,
)

# Val features pour eval_loss (labels) -> on réutilise prepare_train_features
val_features_for_loss = ds["validation"].map(
    lambda x: prepare_train_features(x, tokenizer),
    batched=True,
    remove_columns=ds["validation"].column_names,
)

# Val features pour EM/F1 (offsets + example_id)
val_features_for_pred = ds["validation"].map(
    lambda x: prepare_validation_features(x, tokenizer),
    batched=True,
    remove_columns=ds["validation"].column_names,
)



Map: 100%|█████████████████████████| 1000/1000 [00:00<00:00, 4521.23 examples/s]


## 9) Fine-tuning du modèle (entraînement)

Cette cellule configure l’entraînement et lance le fine-tuning.

- **Entrées :**
  - `train_features` (train)
  - `val_features` (validation)
- **Ce que fait la cellule :**
  1) définit les paramètres d’entraînement (`TrainingArguments`) :
     - batch size, epochs, fp16, sauvegarde par epoch, etc.
  2) crée un `Trainer` avec :
     - le modèle,
     - les données train et val,
  3) lance l’entraînement avec `trainer.train()`.

- **Sorties :**
  - des checkpoints sauvegardés dans `outputs_roberta_qa/`
  - le meilleur checkpoint est rechargé automatiquement à la fin (selon `eval_loss`).

In [ ]:
args = TrainingArguments(
    output_dir="outputs_roberta_qa",
    learning_rate=3e-5,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,

    eval_strategy="epoch",      
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", 
    greater_is_better=False,          

    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_features,
    eval_dataset=val_features_for_loss, 
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

trainer.train()



## (Optionnel) Recharger le modèle déjà entraîné (si `best_model` existe)

Cette cellule sert à **recharger** le modèle sauvegardé pour éviter de refaire l’entraînement.

!!! On exécute cette cellule **uniquement si** le dossier suivant existe :
- `outputs_roberta_qa/best_model/`

### Ce que fait la cellule
1) Charge le **tokenizer** et le **modèle** depuis `outputs_roberta_qa/best_model`.
2) Crée un objet `Trainer` minimal pour pouvoir utiliser ensuite :
- `trainer.predict(...)` (pour calculer EM/F1),
sans relancer `trainer.train()`.

### Résultat
Après cette cellule, on peut continuer le notebook comme si le modèle venait d’être entraîné (uniquement pour la partie prédiction/évaluation).

In [9]:
BEST_DIR = "outputs_roberta_qa/best_model"

tokenizer = AutoTokenizer.from_pretrained(BEST_DIR, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(BEST_DIR)


args = TrainingArguments(
    output_dir="outputs_roberta_qa",
    per_device_eval_batch_size=32,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

/tmp/ipykernel_30476/160596806.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/rayane/Desktop/Paris Cité/M1/PPD/RAG/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## 10) Évaluation sur validation (Exact Match et F1)

Cette cellule calcule les métriques EM et F1 sur la base validation.

- **Entrées :**
  - `val_features` : features validation,
  - `ds["validation"]` : exemples originaux + answers gold.
- **Ce que fait la cellule :**
  1) récupère les logits du modèle avec `trainer.predict(val_features)`,
  2) reconstruit les réponses texte avec `postprocess_qa_predictions`,
  3) calcule les métriques SQuAD avec `evaluate.load("squad")` :
     - `exact_match` et `f1`,
  4) crée un petit tableau de résultats.
- **Sorties :**
  - `results` : dictionnaire avec EM et F1,
  - `df` : tableau récapitulatif affiché.

In [11]:
import accelerate

# Sauvegarde la méthode originale
_orig_unwrap_model = accelerate.Accelerator.unwrap_model

def _unwrap_model_compat(self, model, *args, **kwargs):
    # Transformers récent passe keep_torch_compile=..., mais Accelerate ancien ne le supporte pas
    kwargs.pop("keep_torch_compile", None)
    return _orig_unwrap_model(self, model, *args, **kwargs)

# Patch
accelerate.Accelerator.unwrap_model = _unwrap_model_compat

In [12]:
# Predict on validation (features pour EM/F1)
raw_pred = trainer.predict(val_features_for_pred)
start_logits, end_logits = raw_pred.predictions

# Postprocess into text answers
val_examples = ds["validation"]
val_features_list = [val_features_for_pred[i] for i in range(len(val_features_for_pred))]

preds = postprocess_qa_predictions(
    examples=val_examples,
    features=val_features_list,
    raw_predictions=(start_logits, end_logits),
    tokenizer=tokenizer,
)

# Compute EM/F1 using official SQuAD metric
squad_metric = evaluate.load("squad")
formatted_preds = [{"id": k, "prediction_text": v} for k, v in preds.items()]
formatted_refs = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in val_examples]

results = squad_metric.compute(predictions=formatted_preds, references=formatted_refs)

df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
}])

print("\n=== Validation Results ===")
print(df.to_string(index=False))


=== Validation Results ===
     split  exact_match        f1
validation         89.9 93.827106


## 11) Sauvegarde des résultats et du modèle

Cette cellule sauvegarde les métriques et le modèle pour ne pas avoir à réentraîner plus tard.

### Ce que fait la cellule

1) **Créer le dossier `results/`**
- **But :** s’assurer que le dossier existe avant d’écrire des fichiers.

2) **Sauvegarder le tableau des métriques (CSV)**
- **Fichier créé :** `results/metrics_table.csv`
- **Contenu :** le tableau `df` avec `exact_match` et `f1`.

3) **Sauvegarder les métriques (JSON)**
- **Fichier créé :** `results/metrics.json`
- **Contenu :** le dictionnaire `results` (mêmes métriques, format JSON pratique pour le rapport).

4) **Sauvegarder le meilleur modèle**
- **Dossier créé :** `outputs_roberta_qa/best_model/`
- **But :** recharger directement ce modèle plus tard, sans relancer l’entraînement.
- On sauvegarde aussi le tokenizer dans le même dossier.

### Sorties (fichiers/dossiers créés)
- `results/metrics_table.csv`
- `results/metrics.json`
- `outputs_roberta_qa/best_model/`

In [15]:
# Save
os.makedirs("results", exist_ok=True)
df.to_csv("results/metrics_table_QA.csv", index=False)
print("\nSaved: results/metrics_table_QA.csv")
print("Model checkpoints saved in: outputs_roberta_qa/")

trainer.save_model("outputs_roberta_qa/best_model")
tokenizer.save_pretrained("outputs_roberta_qa/best_model")


Saved: results/metrics_table_QA.csv
Model checkpoints saved in: outputs_roberta_qa/


('outputs_roberta_qa/best_model/tokenizer_config.json',
 'outputs_roberta_qa/best_model/special_tokens_map.json',
 'outputs_roberta_qa/best_model/vocab.json',
 'outputs_roberta_qa/best_model/merges.txt',
 'outputs_roberta_qa/best_model/added_tokens.json',
 'outputs_roberta_qa/best_model/tokenizer.json')

## Exemples sur des questions présentent dans validation avec comparaison gold (réél) et générée

In [14]:
import random
ids = list(preds.keys())
for ex_id in random.sample(ids, 10):
    # gold
    ex = next(e for e in ds["validation"] if e["id"] == ex_id)
    gold = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    print("Q:", ex["question"])
    print("Pred:", preds[ex_id])
    print("Gold:", gold)
    print("---")

Q: In 2011, who was the first player to be chosen in the NFL draft?
Pred: Newton
Gold: Newton
---
Q: In early 2012, Goodell said that Super Bowl 50 would be what?
Pred: an important game for us as a league".
Gold: spectacular
---
Q: Super Bowl 50 determined the NFL champion for what season?
Pred: 2015
Gold: 2015
---
Q: Which player did the Panthers lose to an ACL injury in a preseason game?
Pred: Kelvin Benjamin
Gold: Kelvin Benjamin
---
Q: How many years has Thomas Davis played in the NFL?
Pred: 11
Gold: 11
---
Q: What was the name of the event at the The Embarcadero that was held prior to Super Bowl 50 to help show off some of the things that San Francisco has to offer?
Pred: Super Bowl City
Gold: Super Bowl City
---
Q: When did the Warsaw Uprising begin?
Pred: 1 August 1944
Gold: August 1944
---
Q: On what date did the Florida legislature decide against the plan to renovate the Miami stadium?
Pred: May 3, 2013
Gold: May 3, 2013
---
Q: Which company sponsored a contest called "Small 